In [12]:
import csv
import json
import itertools
import uuid
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, DCTERMS, XSD

# ==========================================================
# CONFIG
# ==========================================================

CSV_PATH = "/Users/padrian/Downloads/gramazio-kohler-archiv-server_DROIDresults(3).csv"
CONFIG_PATH = "/Users/padrian/Documents/08_Tools/27_DCA_Ingest/res/Gantenbein_Ingest.json"
ROW_LIMIT = 100          # None = alles
EXCLUDE_NAMES = {".DS_Store"}

# ==========================================================
# LOAD CONFIG
# ==========================================================

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

BASE_URI = CONFIG["identifiers"]["uri_base"].rstrip("/") + "/"

# ==========================================================
# NAMESPACES
# ==========================================================

RICO = Namespace("https://www.ica.org/standards/RiC/ontology#")
PREMIS = Namespace("http://www.loc.gov/premis/rdf/v3/")
EBCORE = Namespace("http://www.ebu.ch/metadata/ontologies/ebucore/ebucore#")
SCHEMA = Namespace("http://schema.org/")
HASH = Namespace("http://id.loc.gov/vocabulary/preservation/cryptographicHashFunctions/")
DCA_ID = Namespace("https://dca.ethz.ch/id/")

# ==========================================================
# INIT GRAPH
# ==========================================================

g = Graph()
g.bind("rico", RICO)
g.bind("premis", PREMIS)
g.bind("dcterms", DCTERMS)
g.bind("ebucore", EBCORE)
g.bind("schema", SCHEMA)
g.bind("dca-id", DCA_ID)

# ==========================================================
# STEP 1 — READ CSV
# ==========================================================

rows = {}

with open(CSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    iterable = itertools.islice(reader, ROW_LIMIT) if ROW_LIMIT else reader

    for row in iterable:
        if row["NAME"] in EXCLUDE_NAMES:
            continue
        rows[row["ID"]] = row

# ==========================================================
# STEP 2 — RecordResources + HIERARCHY
# ==========================================================

rr_map = {}

for id_, row in rows.items():
    if row["TYPE"] == "Folder":
        rr_uri = DCA_ID[f"rr_{id_}"]
    else:
        _md5 = row["HASH"][:16] if row["HASH"] else uuid.uuid4().hex[:16]
        rr_uri = DCA_ID[f"{_md5}/rr_{id_}"]
    rr_map[id_] = rr_uri

    g.add((rr_uri, RDF.type, RICO.RecordResource))

    if row["TYPE"] == "Folder":
        g.add((rr_uri, RDF.type, RICO.RecordSet))
    else:
        g.add((rr_uri, RDF.type, RICO.Record))

for id_, row in rows.items():
    parent = row["PARENT_ID"]
    if parent and parent in rr_map:
        g.add((rr_map[id_], RICO.isOrWasIncludedIn, rr_map[parent]))

# ==========================================================
# STEP 3 — INSTANTIATION + FILE (Datei‑Kern)
# ==========================================================

pid_counter = 1

for id_, row in rows.items():
    if row["TYPE"] != "File":
        continue

    rr_uri = rr_map[id_]

    inst_uri = URIRef(f"{rr_uri}/i_1")
    file_uri = URIRef(f"{rr_uri}/i_1/b_1")

    # --- Instantiation (primäre Ressource, kein Hash)
    g.add((inst_uri, RDF.type, RICO.Instantiation))
    g.add((inst_uri, RDF.type, PREMIS.Representation))
    g.add((inst_uri, RICO.isInstantiationOf, rr_uri))
    g.add((inst_uri, RICO.name, Literal(row["NAME"])))
    g.add((inst_uri, RICO.hasOrHadCategory,
           URIRef("http://docuteam.ch/vocab/instantiationcategory/file")))

    # --- Identifier (sekundär → Hash‑URI!)
    pid_uri = URIRef(f"{inst_uri}#pid_{uuid.uuid4().hex}")
    g.add((pid_uri, RDF.type, RICO.Identifier))
    g.add((pid_uri, RICO.isOrWasIdentifierOf, inst_uri))
    g.add((pid_uri, RICO.hasIdentifierType,
           URIRef("http://docuteam.ch/vocab/identifiertypes/pid")))
    g.add((pid_uri, RICO.normalizedValue,
           Literal(f"CH-00000-0:{pid_counter}")))
    g.add((pid_uri, SCHEMA.position, Literal(1, datatype=XSD.integer)))
    pid_counter += 1

    # --- premis:File (primäre Ressource, kein Hash)
    g.add((file_uri, RDF.type, PREMIS.File))
    g.add((file_uri, RICO.isOrWasComponentOf, inst_uri))

    if row["SIZE"]:
        g.add((file_uri, PREMIS.size,
               Literal(row["SIZE"], datatype=XSD.integer)))

    g.add((file_uri, PREMIS.storedAt, Literal(row["FILE_PATH"])))

    if row["MIME_TYPE"]:
        g.add((file_uri, EBCORE.hasMimeType, Literal(row["MIME_TYPE"])))

    if row["PUID"]:
        pronom_uri = URIRef(
            f"https://www.nationalarchives.gov.uk/PRONOM/{row['PUID']}"
        )
        g.add((file_uri, DCTERMS.format, pronom_uri))

    if row["HASH"]:
        fixity_uri = URIRef(f"{file_uri}#fixity_{uuid.uuid4().hex}")
        g.add((fixity_uri, RDF.type, HASH.md5))
        g.add((fixity_uri, RDF.value, Literal(row["HASH"])))
        g.add((file_uri, PREMIS.fixity, fixity_uri))

# ==========================================================
# SERIALIZE
# ==========================================================

g.serialize("output.ttl", format="turtle")
print("✅ Matterhorn‑konforme TTL (URI‑Stil korrekt) erzeugt: output.ttl")

✅ Matterhorn‑konforme TTL (URI‑Stil korrekt) erzeugt: output.ttl
